# 11 pamoka - Agentas-agentui (A2A) protokolas


## Nustatymas


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Kas yra A2A protokolas?

**Agentas-agentui (A2A) protokolas** yra atviras standartas, leidžiantis AI agentams bendrauti,
atrasti vieniems kitus ir bendradarbiauti – net kai jie sukurti skirtingose sistemose arba talpinami
skirtingose paslaugose.

Pagrindinės sąvokos:

- **Atranka** – Agentai paskelbia *Agentų kortelę*, kurioje aprašomos jų galimybės, todėl
  kiti agentai (arba koordinuojantys mechanizmai) gali lengvai surasti tinkamą specialistą užduočiai.
- **Žinučių siuntimas** – Agentai keičiasi struktūruotomis žinutėmis per bendrą protokolą, todėl
  vieno agente pateiktas prašymas gali būti suprastas ir įvykdytas kitu, nepaisant vidinės
  įgyvendinimo skirtumų.
- **Užduoties gyvavimo ciklas** – A2A apibrėžia būsenas, tokias kaip *pateikta*, *vykdoma*, *baigta* ir
  *nepavyko*, suteikdamas koordinatoriui pilną matomumą apie patikėtos užduoties eigą.

Šioje pamokoje mes simuliuojame A2A stiliaus bendradarbiavimą, jungdami tris specializuotus kelionių agentus
į procesą, kuriame kiekvienas agentas prisideda savo ekspertize ir perduoda rezultatus kitam.


## Specializuotų kelionių agentų kūrimas


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Daugelio agentų bendradarbiavimas per darbo eigą

Mes sujungiame tris agentus į nuoseklią darbo eigą, kuri atspindi A2A žinučių perdavimą:

1. **CurrencyExchangeAgent** gauna vartotojo užklausą ir pateikia valiutos patarimus.
2. **ActivityPlannerAgent** gauna praturtintą kontekstą ir prideda veiklų rekomendacijas.
3. **TravelManagerAgent** sujungia abu įvestis į galutinį kelionės apžvalgą.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Supratimas apie A2A gamyboje

Gamybos aplinkoje A2A protokolas atveria galingas tarp paslaugų scenarijus:

| Galimybė | Aprašymas |
|---|---|
| **Tarp sistemų bendradarbiavimas** | Agentas, sukurtas vienu karkasu, gali deleguoti užduotis agentui, sukurtam bet kuriuo kitu A2A suderinamu karkasu, leidžiant tikrą tarporganizacinį bendradarbiavimą. |
| **Paslaugų ribos** | Agentai gali gyventi atskiruose mikropaslaugų moduliuose, debesijos regionuose ar net skirtingose organizacijose, ir vis tiek sklandžiai bendradarbiauti. |
| **Dinaminis paieška** | Orkestras gali vykdymo metu užklausti Agentų Kortelių registrą, kad surastų geriausiai tinkantį specialistą konkrečiai uždučių daliai. |
| **Srautinės transliacijos ir pranešimai** | A2A palaiko Serverio siunčiamus įvykius (SSE) realaus laiko pažangos atnaujinimams ir pranešimus ilgai trunkančioms užduotims. |

Aukščiau sukurtas darbas yra supaprastinta šiame procese vykstančio modelio versija. Realioje
diegimo aplinkoje kiekvienas agentas pateiktų HTTP galinį tašką, skelbtų Agentų Kortelę ir bendrautų
per A2A JSON-RPC protokolą.


## Santrauka

Šioje pamokoje sužinojote:

1. **Kas yra A2A protokolas** — atviras standartas agentų tarpusavio atradimui, žinučių siuntimui,
   ir užduočių valdymui.
2. **Kaip sukurti specializuotus agentus** — valiutų keitimo agentą, veiklos planuotojo agentą,
   ir kelionių vadybininko orkestratorių.
3. **Kaip sujungti agentus į darbo eigą** — naudojant `WorkflowBuilder`, modeliuojant sekamą
   žinučių perdavimą tarp agentų.
4. **Kaip A2A veikia gamyboje** — leidžiantį tarpplatforminį, tarp paslaugų bendradarbiavimą
   su dinamišku atradimu ir srautiniais atnaujinimais.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba laikomas autoritetingu šaltiniu. Svarbiai informacijai rekomenduojama naudoti profesionalų žmogiškąjį vertimą. Mes neatsakome už jokius nesusipratimus ar neteisingą interpretaciją, kilusią naudojantis šiuo vertimu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
